In [3]:
!pip install pybaseball

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pybaseball as pb
import csv
!pip install plotly nbformat

In [168]:
from IPython.display import display

## Removes any restrictions on how many rows or columns Jupyter Notebook will display in any given output. 
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# Searches for the player id of desired MLB pitcher.  
player = pb.playerid_lookup("Cease", "Dylan")

# Assigns the variable "mlbam_id" with the unique ID number of the pitcher established in the above^ code (Dylan Cease). 
mlbam_id = player["key_mlbam"].iloc[0]

# Creates the "pitch_data" database for Dylan Cease (using his mlbam_id) from a specific start date to end date. 
pitch_data = pb.statcast_pitcher(start_dt="2024-03-25", end_dt="2026-10-01", player_id=mlbam_id)

# Sorts the datasets I am pulling from pybaseball according game date and pitch number. 
pitch_data = pitch_data.sort_values(["game_date", "at_bat_number", "pitch_number"])

# Creates a dictionary that assigns hits (singles, doubles, etc.) a numerical value. 
tb_dict = {
    "single": 1, 
    "double": 2, 
    "triple": 3, 
    "home_run": 4}

# Creates a list including all hit types. 
hit_events = ["single", "double", "triple", "home_run"]

# Creates the "is_new_pa" truncation that combines pitch-by-pitch data into at-bat data. Identifies when a new player is listed, and combines the previous info and begins a new line (and so on). 
pitch_data["is_new_pa"] = (
    (pitch_data["game_date"] != pitch_data["game_date"].shift())
    | (pitch_data["batter"] != pitch_data["batter"].shift())
    | (pitch_data["at_bat_number"] != pitch_data["at_bat_number"].shift()))

pitch_data["pa_id"] = pitch_data.groupby("game_date")["is_new_pa"].cumsum()

# Implements the "pa_id" truncation into the "pitch_data" chart/database. 
pitch_data["pitch_in_game"] = pitch_data.groupby("game_date").cumcount()
pitch_data["pitch_count_at_pa_start"] = pitch_data.groupby(["game_date", "pa_id"])["pitch_in_game"].transform("min")

# Creates a new column "is_hit" in pitch_data that prints 1 or 0 for hits and non-hits (using the hit_events list). 
pitch_data["is_hit"] = pitch_data["events"].isin(hit_events).astype(int)

# Creates another column "total_bases" in pitch_data that includes the number of total bases corresponding to the at-bat result (using the tb_dict dictionary). 
pitch_data["total_bases"] = pitch_data["events"].map(tb_dict).replace(np.nan, 0)

# Creates a new list "at_bat_events" that updates hit_events with non-hit at-bat results. 
at_bat_events = hit_events + ["strikeout", "field_out", "force_out"]

# Creates another column "is_at_bat" in pitch_data that prints 1 or 0 for at-bats and non-at-bats (using the at_bat_events list). 
pitch_data["is_at_bat"] = pitch_data["events"].isin(at_bat_events).astype(int)

# Creates a new list "on_base" that updates hit_events with non-hit at-bat results that involve the batter getting on base (walks, hbp, etc.). 
on_base = hit_events + ["walk", "hit_by_pitch", "intent_walk"]

# Creates another column "on_base" in pitch_data that prints 1 or 0 for on-base and not-on-base at-bat results (using the on_base list). 
pitch_data["on_base"] = pitch_data["events"].isin(on_base).astype(int)

# Creates another column "plate_app" in pitch_data that prints 1 or 0 for plate appearances and non-plate appearances. 
plate_apps = [0, 1]
pitch_data["plate_app"] = pitch_data["events"].notna().astype(int)

# Creates a plate appearances-aggregation of the chart - separating data entries by plate appearance rather than pitch. 
pa_summary = (pitch_data.groupby(["game_date", "pa_id"]).agg(
        pitch_count_at_pa_start=("pitch_count_at_pa_start", "first"), 
        batter=("batter", "first"),
        result=("events","last"),  
        is_hit=("is_hit", "max"),  
        total_bases=("total_bases", "max"), 
        is_at_bat=("is_at_bat", "max"), 
        on_base=("on_base", "max"), 
        plate_app=("plate_app", "max")).reset_index())

display(pa_summary)

Gathering Player Data


,game_date,pa_id,pitch_count_at_pa_start,batter,result,is_hit,total_bases,is_at_bat,on_base,plate_app
0,2024-03-25,1,0,641487,single,1,1.0,1,1,1
1,2024-03-25,2,3,677594,field_out,0,0.0,1,0,1
2,2024-03-25,3,9,593871,field_out,0,0.0,1,0,1
3,2024-03-25,4,13,641598,field_out,0,0.0,1,0,1
4,2024-03-25,5,17,663728,strikeout,0,0.0,1,0,1
5,2024-03-25,6,20,571745,field_out,0,0.0,1,0,1
6,2024-03-25,7,28,686527,strikeout,0,0.0,1,0,1
7,2024-03-25,8,35,664034,single,1,1.0,1,1,1
8,2024-03-25,9,36,649966,field_out,0,0.0,1,0,1
9,2024-03-25,10,39,641487,walk,0,0.0,0,1,1


In [170]:
# Creates rolling averages for AVG, SLG, OBP, and OPS across each game in the above ^ chart. 
# I want to calculate these stats from a specific plate appearance ONWARD, meaning the stats in a game AFTER a given PA. 
# To calculate that for each PA, I use the formula: Remaining Stats = Total Game Sum - Cumulative Sum Up To Now + Current Row Value

# Calculates the total number of each stat at each plate appearance per game. 
pa_summary["tot_hits"] = pa_summary.groupby("game_date")["is_hit"].cumsum()
pa_summary["tot_total_bases"] = pa_summary.groupby("game_date")["total_bases"].cumsum()
pa_summary["tot_at_bats"] = pa_summary.groupby("game_date")["is_at_bat"].cumsum()
pa_summary["tot_on_base"] = pa_summary.groupby("game_date")["on_base"].cumsum()
pa_summary["tot_plate_app"] = pa_summary.groupby("game_date")["plate_app"].cumsum()

# Calculates the total sum of each stat per game. 
total_hits = pa_summary.groupby("game_date")["is_hit"].transform("sum")
total_tb = pa_summary.groupby("game_date")["total_bases"].transform("sum")
total_ab = pa_summary.groupby("game_date")["is_at_bat"].transform("sum")
total_pa = pa_summary.groupby("game_date")["plate_app"].transform("sum")
total_ob = pa_summary.groupby("game_date")["on_base"].transform("sum")

# Calculates remaining stats (Total - Cumulative + Current)
pa_summary["game_rem_hits"] = total_hits - pa_summary["tot_hits"] + pa_summary["is_hit"]
pa_summary["game_rem_tb"] = total_tb - pa_summary["tot_total_bases"] + pa_summary["total_bases"]
pa_summary["game_rem_ab"] = total_ab - pa_summary["tot_at_bats"] + pa_summary["is_at_bat"]
pa_summary["game_rem_pa"] = total_pa - pa_summary["tot_plate_app"] + pa_summary["plate_app"]
pa_summary["game_rem_ob"] = total_ob - pa_summary["tot_on_base"] + pa_summary["on_base"]

# Calculates the AVG, SLG, OBP, and OPS for the remaining-game stats. 
pa_summary["game_rem_avg"] = (pa_summary["game_rem_hits"] / pa_summary["game_rem_ab"]).fillna(0)
pa_summary["game_rem_slg"] = (pa_summary["game_rem_tb"] / pa_summary["game_rem_ab"]).fillna(0)
pa_summary["game_rem_obp"] = (pa_summary["game_rem_ob"] / pa_summary["game_rem_pa"]).fillna(0)
pa_summary["game_rem_ops"] = pa_summary["game_rem_slg"] + pa_summary["game_rem_obp"]

cols_to_show = ["game_date", "pa_id", "pitch_count_at_pa_start", "game_rem_avg", "game_rem_slg", "game_rem_obp", "game_rem_ops"]

display(pa_summary[cols_to_show])

,game_date,pa_id,pitch_count_at_pa_start,game_rem_avg,game_rem_slg,game_rem_obp,game_rem_ops
0,2024-03-25,1,0,0.294118,0.470588,0.368421,0.839009
1,2024-03-25,2,3,0.250000,0.437500,0.333333,0.770833
2,2024-03-25,3,9,0.266667,0.466667,0.352941,0.819608
3,2024-03-25,4,13,0.285714,0.500000,0.375000,0.875000
4,2024-03-25,5,17,0.307692,0.538462,0.400000,0.938462
5,2024-03-25,6,20,0.333333,0.583333,0.428571,1.011905
6,2024-03-25,7,28,0.363636,0.636364,0.461538,1.097902
7,2024-03-25,8,35,0.400000,0.700000,0.500000,1.200000
8,2024-03-25,9,36,0.333333,0.666667,0.454545,1.121212
9,2024-03-25,10,39,0.375000,0.750000,0.500000,1.250000


In [98]:
# Uses .merge() to merge the rem_ops from pa_summary onto pitch_data. 
# pitch_level_curve = pitch_data.merge(pa_summary[["game_date", "pa_id", "game_rem_ops"]], on=["game_date", "pa_id"], how="left")
# pitch_level_curve["game_rem_ops"] = (pitch_level_curve.groupby("game_date")["game_rem_ops"].ffill())

# season_pitch_profile = (pitch_level_curve.groupby("pitch_in_game").agg(avg_rem_ops=("game_rem_ops", "mean").reset_index()
#                                                                        .rename(columns={"pitch_in_game": "pitch_count"})))

In [174]:
pa_summary["prev_pa_id"] = pa_summary.groupby("game_date")["pa_id"].shift(1).fillna(0)
pa_summary["prev_pitch_count"] = pa_summary.groupby("game_date")["pitch_count_at_pa_start"].shift(1).fillna(0)

# Creates a new column: ops_jump, which is the difference in ops between the current and previous plate appearances. 
pa_summary["ops_jump"] = pa_summary.groupby("game_date")["game_rem_ops"].diff()

# Filters the jumps to only include pitch counts above 70 and OPS jumps greater than 0.3. 
filtered_jumps = pa_summary[(pa_summary["pitch_count_at_pa_start"] >= 70) & (pa_summary["ops_jump"] > 0.3)]

# Calculates the biggest jump per game using .loc() and .idxmax(). 
max_jumps_per_game = filtered_jumps.loc[filtered_jumps.groupby("game_date")["ops_jump"].idxmax()]

display(max_jumps_per_game[["game_date", "prev_pa_id", "pa_id", "prev_pitch_count", "pitch_count_at_pa_start", "ops_jump"]])

# Calculates the overall opponent OPS-per-pitch spike by taking the average across all the starts in a season. 
overall_avg_pitch_spike = max_jumps_per_game["pitch_count_at_pa_start"].mean()

print(f"{overall_avg_pitch_spike:.1f}")

# Creates a weight according to how big the jump in OPS is. 
weights = max_jumps_per_game["ops_jump"]

# Calculates the same overall stat, weighted according to how big the OPS jumps were. 
weighted_avg_pitch = (max_jumps_per_game["pitch_count_at_pa_start"] * weights).sum() / weights.sum()

print(f"{weighted_avg_pitch:.1f}")

# display(pa_summary)

,game_date,prev_pa_id,pa_id,prev_pitch_count,pitch_count_at_pa_start,ops_jump
223,2024-05-14,19.0,20,86.0,89,0.600000
278,2024-05-25,26.0,27,93.0,98,0.666667
297,2024-05-31,17.0,18,73.0,76,0.533333
349,2024-06-10,25.0,26,88.0,93,0.333333
369,2024-06-16,17.0,18,71.0,74,0.566667
462,2024-07-07,23.0,24,88.0,90,0.833333
608,2024-08-17,21.0,22,83.0,88,1.416667
638,2024-08-22,27.0,28,93.0,97,0.500000
662,2024-08-27,23.0,24,86.0,91,1.000000
680,2024-09-01,17.0,18,67.0,70,0.750000


87.2
88.3


In [44]:
# first_time_thru = pa_summary.iloc[:9, ]
# display(first_time_thru)

# late_innings = pa_summary[pa_summary["pitch"] > 90]
# display(late_innings)

# window_size = 10

# roll_hits = pa_summary["is_hit"].rolling(window=window_size, min_periods=1).sum()
# roll_ab = pa_summary["is_at_bat"].rolling(window=window_size, min_periods=1).sum()
# roll_tb = pa_summary["total_bases"].rolling(window=window_size, min_periods=1).sum()
# roll_ob = pa_summary["on_base"].rolling(window=window_size, min_periods=1).sum()
# roll_pa = pa_summary["plate_app"].rolling(window=window_size, min_periods=1).sum()

# rolling_slg = (roll_tb / roll_ab).fillna(0)
# rolling_obp = (roll_ob / roll_pa).fillna(0)

# pa_summary["rolling_ops"] = rolling_slg + rolling_obp

# pa_summary["rolling_ba"] = (roll_hits / roll_ab).fillna(0)

# display(pa_summary)

In [56]:
# I use this chunk of code to identify specific MLB pitchers and find the list of starts they made in a given season so that I can choose one start to analyze. 

player = pb.playerid_lookup("cease", "dylan")
mlbam_id = player["key_mlbam"].iloc[0]

df_games = pb.statcast_pitcher(start_dt='2026-03-25', end_dt='2026-10-01', player_id=mlbam_id)

print("Game Dates Pitched:", df_games['game_date'].unique())

Gathering Player Data
Game Dates Pitched: ['2026-09-18' '2026-09-13' '2026-09-07' '2026-09-02' '2026-08-28'
 '2026-08-22' '2026-08-16' '2026-08-11' '2026-08-06' '2026-07-31'
 '2026-07-25' '2026-07-20' '2026-07-08' '2026-07-03' '2026-06-27'
 '2026-06-22' '2026-06-16' '2026-06-09' '2026-05-24' '2026-05-19'
 '2026-05-13' '2026-05-08' '2026-05-02' '2026-04-27' '2026-04-20'
 '2026-04-15' '2026-04-08' '2026-04-03' '2026-03-28']
